# Clase 5 — Transformers y el mecanismo de atención

## Objetivos de aprendizaje

- Entender los conceptos de Query (Q), Key (K) y Value (V) en self-attention.
- Ver cómo se calculan las puntuaciones de atención (producto punto, escalado, softmax).
- Implementar una capa de self-attention personalizada en Keras.
- Construir y entrenar un modelo encoder-decoder (LSTM + atención) para traducción EN→ES.
- Entender por qué este modelo NO es el Transformer canónico del paper original.

## Qué vamos a construir

Un mini traductor de inglés a español entrenado con solo 5 frases de ejemplo.
No va a traducir nada útil en la vida real (el dataset es demasiado pequeño),
pero nos deja ver, con código que se ejecuta de verdad, cómo funciona por
dentro la atención.

## ⚠️ Aviso importante antes de empezar

El material original de este notebook se titula "Transformers with Keras",
pero **no implementa el Transformer del paper "Attention Is All You Need"**
(el que usan de verdad GPT, Llama o BERT). Lo que vamos a construir es un
modelo híbrido más sencillo:

- Un **encoder** y un **decoder** basados en **LSTM** (recurrente), no en
  atención pura.
- Una única capa de **self-attention personalizada** que conecta encoder y
  decoder — sin atención multi-cabeza, sin *positional encoding*.

Es la introducción conceptual perfecta a Q/K/V, pero no confundas esta
arquitectura con la de un LLM real. Más detalles en el capítulo
[08_transformers_atencion.md](../manual/08_transformers_atencion.md).

## Apunte rápido: ¿qué son Q, K y V?

- **Query (Q)**: la pregunta que hace cada palabra: "¿quién me ayuda a
  entender mi contexto?".
- **Key (K)**: la etiqueta con la que cada palabra se presenta ante las demás.
- **Value (V)**: la información real que aporta esa palabra.

Cálculo de la atención: `scores = (Q · Kᵀ) / √dk`, luego `softmax(scores)`
para obtener pesos que suman 1, y esos pesos ponderan los Values.

## Paso 0: instalar dependencias

Ejecuta esta celda una sola vez (usa las mismas versiones del material fuente).

In [ ]:
!pip install tensorflow_cpu==2.17.1
!pip install matplotlib==3.9.2

print("==== Librerías instaladas =====")

In [ ]:
import os
# Suprime mensajes de advertencia de TensorFlow por usar CPU
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate, Layer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter('ignore', FutureWarning)

## Paso 1: preparar los datos

Definimos 5 pares de frases inglés-español. A cada frase en español le
añadimos los tokens especiales `startseq` / `endseq`, que le indican al
decoder cuándo empezar y cuándo terminar de traducir.

In [ ]:
# Pares de frases paralelas (inglés -> español)
input_texts = [
    "Hello.", "How are you?", "I am learning machine translation.",
    "What is your name?", "I love programming."
]
target_texts = [
    "Hola.", "¿Cómo estás?", "Estoy aprendiendo traducción automática.",
    "¿Cuál es tu nombre?", "Me encanta programar."
]

# startseq/endseq le dicen al decoder cuándo empezar y cuándo parar
target_texts = ["startseq " + x + " endseq" for x in target_texts]

In [ ]:
# Tokenización: convertimos palabras en números
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

output_tokenizer = Tokenizer()
output_tokenizer.fit_on_texts(target_texts)
output_sequences = output_tokenizer.texts_to_sequences(target_texts)

input_vocab_size = len(input_tokenizer.word_index) + 1
output_vocab_size = len(output_tokenizer.word_index) + 1

print("Vocabulario inglés:", input_vocab_size, "palabras")
print("Vocabulario español:", output_vocab_size, "palabras")

In [ ]:
# Padding: todas las secuencias deben tener la misma longitud
max_input_length = max([len(seq) for seq in input_sequences])
max_output_length = max([len(seq) for seq in output_sequences])

input_sequences = pad_sequences(input_sequences, maxlen=max_input_length, padding='post')
output_sequences = pad_sequences(output_sequences, maxlen=max_output_length, padding='post')

# El decoder recibe la frase española desplazada un paso (entrada / objetivo)
decoder_input_data = output_sequences[:, :-1]
decoder_output_data = output_sequences[:, 1:]

# Convertimos el objetivo a one-hot (una probabilidad por palabra del vocabulario)
decoder_output_data = np.array([np.eye(output_vocab_size)[seq] for seq in decoder_output_data])

## Paso 2: la capa de self-attention

En `build()` inicializamos las matrices entrenables `Wq`, `Wk`, `Wv`. En
`call()` calculamos Q, K, V, las puntuaciones de atención escaladas, el
softmax, y la salida final ponderada por V.

In [ ]:
class SelfAttention(Layer):
    def __init__(self, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        # input_shape es una lista: [forma_de_q, forma_de_k, forma_de_v]
        feature_dim = input_shape[0][-1]

        self.Wq = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='Wq'
        )
        self.Wk = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='Wk'
        )
        self.Wv = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='Wv'
        )
        super(SelfAttention, self).build(input_shape)

    def call(self, inputs):
        # inputs es una lista [query, key, value]
        q, k, v = inputs

        q = K.dot(q, self.Wq)
        k = K.dot(k, self.Wk)
        v = K.dot(v, self.Wv)

        # Atención escalada: (Q . K^T) / sqrt(dk)
        scores = K.batch_dot(q, k, axes=[2, 2])
        dk = K.cast(K.shape(k)[-1], dtype=K.floatx())
        scores = scores / K.sqrt(dk)

        # softmax normaliza las puntuaciones en pesos que suman 1
        attention_weights = K.softmax(scores, axis=-1)
        output = K.batch_dot(attention_weights, v)

        return output

## Paso 3: arquitectura encoder-decoder + atención

- **Encoder**: Embedding + LSTM sobre la frase en inglés. Genera un
  "resumen" (estados `state_h`, `state_c`).
- **Decoder**: Embedding + LSTM sobre la frase en español, inicializado con
  el resumen del encoder.
- **Atención**: el decoder "consulta" las salidas del encoder en cada paso.
- **Dense final**: predice la siguiente palabra en español (softmax sobre
  todo el vocabulario).

In [ ]:
# --- Encoder ---
encoder_inputs = Input(shape=(max_input_length,))
encoder_embedding = Embedding(input_vocab_size, 256)(encoder_inputs)
encoder_lstm = LSTM(256, return_sequences=True, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# --- Decoder ---
decoder_inputs = Input(shape=(max_output_length - 1,))
decoder_embedding = Embedding(output_vocab_size, 256)(decoder_inputs)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

# --- Atención: el decoder consulta las salidas del encoder ---
self_attention = SelfAttention()
attention_output = self_attention([decoder_outputs, encoder_outputs, encoder_outputs])

# Combinamos la salida del decoder con el contexto de atención
decoder_concat = Concatenate(axis=-1)([decoder_outputs, attention_output])

# Capa final: predice la palabra en español
decoder_dense = Dense(output_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_concat)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

## Paso 4: entrenar el modelo

Con un dataset de 5 frases, 100 épocas se entrenan en segundos en la CPU
gratuita de Colab — un número de épocas perfectamente razonable aquí porque
el dataset es minúsculo (no lo hagas con datasets grandes sin GPU).

In [ ]:
history_glorot_adam = model.fit(
    [input_sequences, decoder_input_data],
    decoder_output_data,
    epochs=100,
    batch_size=16
)

## Paso 5: graficar la pérdida de entrenamiento

In [ ]:
plt.plot(history_glorot_adam.history['loss'])
plt.title('Pérdida de entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.show()

## Resumen de la clase

- Self-attention se calcula con tres matrices entrenables `Wq`, `Wk`, `Wv`.
- Las puntuaciones de atención son `(Q · Kᵀ) / √dk`, normalizadas con softmax.
- Este modelo combina LSTM (encoder-decoder) con una capa de atención — **no**
  es el Transformer canónico del paper original.
- La pérdida (loss) debería bajar de forma consistente a lo largo de las 100
  épocas, señal de que el modelo está aprendiendo a asociar las frases.

## Para seguir

- Repasa la teoría completa en
  [manual/08_transformers_atencion.md](../manual/08_transformers_atencion.md).
- Practica lo aprendido en
  [clase05_practica_transformers_atencion.ipynb](../notebooks_practica/clase05_practica_transformers_atencion.ipynb),
  donde vas a comparar el efecto de cambiar el inicializador de los pesos.